
# LightGBM Pathogen Suppression Predictor

Three versions controlled by the VERSION variable below.
Change VERSION to 1, 2, or 3 to switch between feature sets.

VERSION 1 — Full model:          species (100) + R0 (20) + active (20) + size + diversity  [142 features] <br>
VERSION 2 — Nutrient + size:     R0 (20) + active (20) + size + diversity                  [42  features] <br>
VERSION 3 — Pure nutrient:       R0 (20) + active (20)                                     [40  features] <br>

Input:  community_index_v8_backup.csv <br>
Target: community_suppression (float in [0, 1])


In [1]:

VERSION = 1


import pandas as pd
import numpy as np
import lightgbm as lgb


from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

In [2]:


df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated01\community_index_v8_backup.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 30,000 rows × 168 columns


In [3]:

r0_cols     = [f"R0_c{k}"     for k in range(1, 21)]   # 20 concentration cols
active_cols = [f"active_c{k}" for k in range(1, 21)]   # 20 binary flag cols


if VERSION == 1:
    # Full model: species membership + all nutrient info + structural
    species_cols = [f"Species_{i:04d}" for i in range(2, 102)]  # 100 binary cols
    feature_cols = species_cols + r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Full (species + nutrients + size)"

elif VERSION == 2:
    # Nutrient + size: drop species membership, keep community_size
    feature_cols = r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Nutrients + size (no species)"

elif VERSION == 3:
    # Pure nutrient: only raw chemistry — no species, no community_size
    # diversity_level dropped too: it is redundant since XGBoost/LightGBM
    # can infer it directly by counting non-zero R0 columns.
    feature_cols = r0_cols + active_cols
    version_label = "Pure nutrients only (no species, no size)"

else:
    raise ValueError("VERSION must be 1, 2, or 3")

print(f"\nVersion: {version_label}")
print(f"Features: {len(feature_cols)}")


missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found in file. First 3: {missing[:3]}\n"
                   f"Make sure you are loading community_index_v8_backup.csv (30,000 rows), "
                   f"not community_dataset_v8_backup.csv (792,000 rows).")

X = df[feature_cols]
y = df['community_suppression']
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")


Version: Full (species + nutrients + size)
Features: 142
X: (30000, 142)   y range: [0.000, 0.999]


In [4]:

strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)
print(f"\nTrain: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")


Train: 24,000   Test: 6,000


In [5]:

scaler = StandardScaler()
ridge  = Ridge(alpha=1.0).fit(scaler.fit_transform(X_train), y_train)
yp_ridge  = ridge.predict(scaler.transform(X_test))
r2_ridge  = r2_score(y_test, yp_ridge)
mae_ridge = mean_absolute_error(y_test, yp_ridge)
print(f"\nRidge R²={r2_ridge:.4f}  MAE={mae_ridge:.4f}")


Ridge R²=0.7708  MAE=0.1110


In [6]:
# lgbm initial fit

early_stop = lgb.early_stopping(stopping_rounds=30, verbose=False)
# lgb.early_stopping: stops training if validation metric hasn't improved
# for `stopping_rounds` consecutive rounds. Returns a callback object.

log_eval = lgb.log_evaluation(period=0)
# lgb.log_evaluation(period=0): suppresses all per-round console output.
# Set period=50 if you want to see progress every 50 rounds.

model_init = lgb.LGBMRegressor(
    n_estimators=500,
    num_leaves=63,

    max_depth=-1,
    learning_rate=0.05,
    subsample=0.8,
    bagging_freq=1,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# LightGBM eval_set syntax is the same as XGBoost.
model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[early_stop, log_eval]
)

yp_init  = model_init.predict(X_test)
r2_init  = r2_score(y_test, yp_init)
mae_init = mean_absolute_error(y_test, yp_init)
print(f"LightGBM initial:  R²={r2_init:.4f}  MAE={mae_init:.4f}  "
      f"best_iter={model_init.best_iteration_}")

LightGBM initial:  R²=0.9587  MAE=0.0355  best_iter=497


In [7]:
# hyperparameter tuning

param_grid = {
    'num_leaves':      [31, 63, 127],
    # The primary complexity knob in LightGBM.
    # 31 = shallow/conservative, 127 = deep/expressive.

    'learning_rate':   [0.01, 0.05, 0.1],

    'subsample':       [0.7, 0.8, 1.0],
    # Note: subsample only works when bagging_freq > 0 (set in base_est below).

    'colsample_bytree':[0.6, 0.8, 1.0],
}


cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=-1,
    bagging_freq=1,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 81 candidates, totalling 243 fits

Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'num_leaves': 127, 'subsample': 1.0}
Best CV RMSE: 0.0652


In [8]:
# final model

model_final = lgb.LGBMRegressor(
    **grid_search.best_params_,
    n_estimators=1000,
    max_depth=-1,
    bagging_freq=1,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

early_stop_final = lgb.early_stopping(stopping_rounds=40, verbose=False)
log_eval_final   = lgb.log_evaluation(period=50)

model_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[early_stop_final, log_eval_final]
)

yp_final   = model_final.predict(X_test)
r2_final   = r2_score(y_test, yp_final)
mae_final  = mean_absolute_error(y_test, yp_final)
rmse_final = np.sqrt(mean_squared_error(y_test, yp_final))

print(f"\nLightGBM final:  R²={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration_}")

[50]	valid_0's l2: 0.004558
[100]	valid_0's l2: 0.00398285
[150]	valid_0's l2: 0.00382221
[200]	valid_0's l2: 0.00373902
[250]	valid_0's l2: 0.00372378
[300]	valid_0's l2: 0.00368423
[350]	valid_0's l2: 0.00367422
[400]	valid_0's l2: 0.00366486
[450]	valid_0's l2: 0.00365235
[500]	valid_0's l2: 0.00364706

LightGBM final:  R²=0.9652  MAE=0.0321  RMSE=0.0604
Trees used: 502


In [9]:
# per condition breakdown

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

print("\nBy diversity level:")
for d, grp in results_df.groupby('diversity_level'):
    print(f"  D={d:<4} R²={r2_score(grp.y_true, grp.y_pred):.4f}  "
          f"MAE={mean_absolute_error(grp.y_true, grp.y_pred):.4f}  n={len(grp):,}")

print("\nBy community size:")
for n, grp in results_df.groupby('community_size'):
    print(f"  N={n:<4} R²={r2_score(grp.y_true, grp.y_pred):.4f}  "
          f"MAE={mean_absolute_error(grp.y_true, grp.y_pred):.4f}  n={len(grp):,}")


By diversity level:
  D=1    R²=0.9914  MAE=0.0264  n=1,200
  D=2    R²=0.9067  MAE=0.0514  n=1,200
  D=4    R²=0.9311  MAE=0.0354  n=1,200
  D=8    R²=0.9684  MAE=0.0280  n=1,200
  D=16   R²=0.9792  MAE=0.0193  n=1,200

By community size:
  N=12   R²=0.9191  MAE=0.0568  n=1,200
  N=16   R²=0.9595  MAE=0.0393  n=1,200
  N=24   R²=0.9766  MAE=0.0271  n=1,200
  N=32   R²=0.9821  MAE=0.0230  n=1,200
  N=48   R²=0.9929  MAE=0.0144  n=1,200


In [10]:
# shap analysis

print('shap analysis')

explainer   = shap.TreeExplainer(model_final)
rng         = np.random.default_rng(42)
idx         = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])
mean_shap   = pd.Series(np.abs(shap_values).mean(0), index=feature_cols).sort_values(ascending=False)

print(f"\nTop 15 features by mean |SHAP|:")
print(mean_shap.head(15).round(5).to_string())

shap analysis

Top 15 features by mean |SHAP|:
R0_c8             0.15655
R0_c1             0.09283
community_size    0.03875
R0_c12            0.02545
Species_0055      0.02374
R0_c19            0.02065
Species_0071      0.01847
Species_0101      0.01605
R0_c14            0.01423
Species_0050      0.01362
Species_0093      0.01318
R0_c3             0.01211
R0_c6             0.01149
Species_0086      0.01062
Species_0057      0.01049


In [11]:
# plots

output_name = f"lgbm_v{VERSION}_results_01.png"
# Output filename automatically reflects the version so you won't overwrite results.

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'LightGBM — {version_label} — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}',
    fontsize=12, fontweight='bold', y=0.99
)

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression'); ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - yp_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)
top_n    = min(25, len(feature_cols))
top_shap = mean_shap.head(top_n)
def feat_col(f):
    if 'Species' in f: return '#2196F3'
    if 'R0'      in f: return '#4CAF50'
    if 'active'  in f: return '#FF9800'
    return '#9C27B0'
cols = [feat_col(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP|'); ax3.set_title(f'Top {top_n} Features (SHAP)')
ax3.legend(handles=[
    Patch(fc='#2196F3', label='Species'),
    Patch(fc='#4CAF50', label='R0'),
    Patch(fc='#FF9800', label='Active'),
    Patch(fc='#9C27B0', label='Structural'),
], loc='lower right', fontsize=7)

ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level'); ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level'); ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N'); ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size'); ax5.set_ylim(0, 1.05); ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
# Group SHAP totals dynamically based on whichever features are present
groups, values, colours = [], [], []
sp_s = mean_shap[[f for f in feature_cols if 'Species' in f]].sum()
if sp_s > 0:
    groups.append('Species'); values.append(sp_s); colours.append('#2196F3')
r0_s = mean_shap[r0_cols].sum()
groups.append('R0'); values.append(r0_s); colours.append('#4CAF50')
ac_s = mean_shap[active_cols].sum()
groups.append('Active'); values.append(ac_s); colours.append('#FF9800')
st_s = mean_shap[[f for f in feature_cols if f in ['community_size','diversity_level']]].sum()
if st_s > 0:
    groups.append('Structural'); values.append(st_s); colours.append('#9C27B0')
ax6.pie(values, labels=groups, colors=colours,
        autopct='%.1f%%', startangle=90, textprops={'fontsize': 10})
ax6.set_title('SHAP by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")


Plot saved: lgbm_v1_results_01.png


In [ ]:
# save outputs

model_save  = f"lgbm_v{VERSION}_model.txt"
pred_save   = f"lgbm_v{VERSION}_predictions.csv"

# LightGBM saves in its own text format (not JSON like XGBoost).
# Reload with: m = lgb.Booster(model_file='lgbm_v1_model.txt')
model_final.booster_.save_model(model_save)

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'residual':        y_test.values - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size':  results_df['community_size'].values,
}).to_csv(pred_save, index=False)

print(f"Model saved:       {model_save}")
print(f"Predictions saved: {pred_save}")

In [12]:
# summary

print(f"\n{'='*60}")
print(f"SUMMARY — LightGBM VERSION {VERSION}: {version_label}")
print(f"{'='*60}")
print(f"Features:  {len(feature_cols)}")
print(f"R²:        {r2_final:.4f}")
print(f"MAE:       {mae_final:.4f}")
print(f"RMSE:      {rmse_final:.4f}")
print(f"Trees:     {model_final.best_iteration_}")
print(f"\nTop 5 features:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat:<25} {val:.5f}")


SUMMARY — LightGBM VERSION 1: Full (species + nutrients + size)
Features:  142
R²:        0.9652
MAE:       0.0321
RMSE:      0.0604
Trees:     502

Top 5 features:
  R0_c8                     0.15655
  R0_c1                     0.09283
  community_size            0.03875
  R0_c12                    0.02545
  Species_0055              0.02374
